In [ ]:
pip install gurobipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 25.1 MB/s eta 0:00:00


#Optimisation Code

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Parameters and Indices
n = 11  # Number of Points (node 0 is warehouse, 10 stops)
K = 4  # Number of Products
np.random.seed(6)

# Base travel distances between specific nodes (in km)
base_travel_distance = {
    (0, 1): 55, (0, 2): 50, (0, 3): 45, (0, 4): 40, (0, 5): 35,
    (0, 6): 60, (0, 7): 65, (0, 8): 70, (0, 9): 75, (0, 10): 80,
    (1, 0): 55, (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60, (1, 6): 65,
    (1, 7): 70, (1, 8): 75, (1, 9): 80, (1, 10): 85,
    (2, 0): 50, (2, 1): 30, (2, 3): 35, (2, 4): 45, (2, 5): 55, (2, 6): 50,
    (2, 7): 55, (2, 8): 60, (2, 9): 65, (2, 10): 70,
    (3, 0): 45, (3, 1): 40, (3, 2): 35, (3, 4): 25, (3, 5): 30, (3, 6): 40,
    (3, 7): 45, (3, 8): 50, (3, 9): 55, (3, 10): 60,
    (4, 0): 40, (4, 1): 50, (4, 2): 45, (4, 3): 25, (4, 5): 20, (4, 6): 35,
    (4, 7): 40, (4, 8): 45, (4, 9): 50, (4, 10): 55,
    (5, 0): 35, (5, 1): 60, (5, 2): 55, (5, 3): 30, (5, 4): 20, (5, 6): 25,
    (5, 7): 30, (5, 8): 35, (5, 9): 40, (5, 10): 45,
    (6, 0): 60, (6, 1): 65, (6, 2): 50, (6, 3): 40, (6, 4): 35, (6, 5): 25,
    (6, 7): 20, (6, 8): 25, (6, 9): 30, (6, 10): 35,
    (7, 0): 65, (7, 1): 70, (7, 2): 55, (7, 3): 45, (7, 4): 40, (7, 5): 30,
    (7, 6): 20, (7, 8): 15, (7, 9): 20, (7, 10): 25,
    (8, 0): 70, (8, 1): 75, (8, 2): 60, (8, 3): 50, (8, 4): 45, (8, 5): 35,
    (8, 6): 25, (8, 7): 15, (8, 9): 10, (8, 10): 15,
    (9, 0): 75, (9, 1): 80, (9, 2): 65, (9, 3): 55, (9, 4): 50, (9, 5): 40,
    (9, 6): 30, (9, 7): 20, (9, 8): 10, (9, 10): 10,
    (10, 0): 80, (10, 1): 85, (10, 2): 70, (10, 3): 60, (10, 4): 55, (10, 5): 45,
    (10, 6): 35, (10, 7): 25, (10, 8): 15, (10, 9): 10,
}

base_travel_time = {k: v / 30 for k, v in base_travel_distance.items()}
travel_times = {(i, j): base_travel_time.get((i, j), 0) + np.random.uniform(-0.5, 0.5)
                for i in range(n) for j in range(n)}

delays = [np.random.uniform(0.1, 0.2) if np.random.rand() < 0.1 else 0 for _ in range(n)]

base_temperature = {'Apples': 5, 'Bananas': 13, 'Tomatoes': 10, 'xyz': 8}
required_temperature = [5, 13, 10, 8]
temperature_profile = [base_temperature['Apples'] * (1 + np.random.uniform(-0.05, 0.05)),
                       base_temperature['Bananas'] * (1 + np.random.uniform(-0.05, 0.05)),
                       base_temperature['Tomatoes'] * (1 + np.random.uniform(-0.05, 0.05)),
                       base_temperature['xyz'] * (1 + np.random.uniform(-0.05, 0.05))]

Q = np.random.uniform(0.5, 1, K)
P = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
alpha = 0.5

model = gp.Model('Perishable Produce Transportation')
x = model.addVars(n, n, vtype=GRB.BINARY, name="x")
u = model.addVars(n, lb=0, ub=n-1, vtype=GRB.CONTINUOUS)

model.addConstr(sum(x[0, j] for j in range(1, n)) == 1)
model.addConstr(sum(x[i, 0] for i in range(1, n)) == 1)
model.addConstrs((sum(x[i, j] for j in range(n) if j != i) == 1 for i in range(1, n)))
model.addConstrs((sum(x[j, i] for j in range(n) if j != i) == 1 for i in range(1, n)))
model.addConstrs((x[i, i] == 0 for i in range(n)))
model.addConstrs((u[i] - u[j] + n * x[i, j] <= n - 1 for i in range(1, n) for j in range(1, n) if i != j))

cost_term = sum((travel_times.get((i, j), 0) + delays[i]) * x[i, j] for i in range(n) for j in range(n))
temperature_penalty = sum(
    x[i, j] * sum(abs(temperature_profile[k] - required_temperature[k]) for k in range(K))
    for i in range(n) for j in range(n)
)

model.setObjective(cost_term + 0.05 * temperature_penalty, GRB.MINIMIZE)

model.optimize()

if model.status == GRB.OPTIMAL:
    print("Optimal objective value:", model.objVal)
    for i in range(n):
        for j in range(n):
            if x[i, j].X > 0.5:
                print(f"Route from {i} to {j}")
else:
    print("No optimal solution found.")






Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 123 rows, 132 columns and 501 nonzeros
Model fingerprint: 0xb5202f6a
Variable types: 11 continuous, 121 integer (121 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [5e-02, 3e+00]
  Bounds range     [1e+00, 1e+01]
  RHS range        [1e+00, 1e+01]
Found heuristic solution: objective 18.3409751
Presolve removed 11 rows and 12 columns
Presolve time: 0.01s
Presolved: 112 rows, 120 columns, 986 nonzeros
Variable types: 10 continuous, 110 integer (110 binary)

Root relaxation: objective 7.905423e+00, 34 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0 

# Reinforcement Learning Code

In [ ]:
import numpy as np
import pandas as pd

num_entries = 2000  # Number of dataset entries
K = 3  # Number of products

# Base travel distances between specific stop pairs
base_travel_distance = {
    (0, 1): 55, (0, 2): 50, (0, 3): 45, (0, 4): 40, (0, 5): 35,
    (0, 6): 60, (0, 7): 65, (0, 8): 70, (0, 9): 75, (0, 10): 80,
    (1, 0): 55, (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60, (1, 6): 65,
    (1, 7): 70, (1, 8): 75, (1, 9): 80, (1, 10): 85,
    (2, 0): 50, (2, 1): 30, (2, 3): 35, (2, 4): 45, (2, 5): 55, (2, 6): 50,
    (2, 7): 55, (2, 8): 60, (2, 9): 65, (2, 10): 70,
    (3, 0): 45, (3, 1): 40, (3, 2): 35, (3, 4): 25, (3, 5): 30, (3, 6): 40,
    (3, 7): 45, (3, 8): 50, (3, 9): 55, (3, 10): 60,
    (4, 0): 40, (4, 1): 50, (4, 2): 45, (4, 3): 25, (4, 5): 20, (4, 6): 35,
    (4, 7): 40, (4, 8): 45, (4, 9): 50, (4, 10): 55,
    (5, 0): 35, (5, 1): 60, (5, 2): 55, (5, 3): 30, (5, 4): 20, (5, 6): 25,
    (5, 7): 30, (5, 8): 35, (5, 9): 40, (5, 10): 45,
    (6, 0): 60, (6, 1): 65, (6, 2): 50, (6, 3): 40, (6, 4): 35, (6, 5): 25,
    (6, 7): 20, (6, 8): 25, (6, 9): 30, (6, 10): 35,
    (7, 0): 65, (7, 1): 70, (7, 2): 55, (7, 3): 45, (7, 4): 40, (7, 5): 30,
    (7, 6): 20, (7, 8): 15, (7, 9): 20, (7, 10): 25,
    (8, 0): 70, (8, 1): 75, (8, 2): 60, (8, 3): 50, (8, 4): 45, (8, 5): 35,
    (8, 6): 25, (8, 7): 15, (8, 9): 10, (8, 10): 15,
    (9, 0): 75, (9, 1): 80, (9, 2): 65, (9, 3): 55, (9, 4): 50, (9, 5): 40,
    (9, 6): 30, (9, 7): 20, (9, 8): 10, (9, 10): 10,
    (10, 0): 80, (10, 1): 85, (10, 2): 70, (10, 3): 60, (10, 4): 55, (10, 5): 45,
    (10, 6): 35, (10, 7): 25, (10, 8): 15, (10, 9): 10,
}

# Add symmetric distances
symmetric_travel_distance = {}
for (i, j), dist in base_travel_distance.items():
    symmetric_travel_distance[(i, j)] = dist
    symmetric_travel_distance[(j, i)] = dist  # Make symmetric

base_travel_distance = symmetric_travel_distance


# Calculate base travel times in seconds
base_travel_time = {k: v / 30*3600  for k, v in base_travel_distance.items()}  # Base time at 50 km/h
base_delay_time = 0  # Base delay time in seconds

# Delay reasons list
delay_reasons = ["Road Closed", "Traffic", "Accident", "Weather", "No Delay"]

# Lists to store generated data
travel_distance_pattern = []
travel_time_pattern = []
delay_time_pattern = []
stops_i = []
stops_j = []
delay_events = []

# Generate shelf life and quality factors
SL_0 = 700 * np.random.rand(K)  # Initial shelf life for each product
SL_r = 100 * np.random.rand(K)  # Required shelf life at destination for each product
Q = np.random.uniform(0.5, 1, K)  # Quality reduction factor for each product

for _ in range(num_entries):
    stop_i = np.random.randint(0, 11)  # Random stop 1-5
    stop_j = np.random.randint(0, 11)

    # Ensure stop_i != stop_j
    while stop_i == stop_j:
        stop_j = np.random.randint(1, 11)

    # Use base travel distance and add some variation
    if (stop_i, stop_j) in base_travel_distance:
        distance = base_travel_distance[(stop_i, stop_j)] + np.random.uniform(-5, 5)
        time = base_travel_time[(stop_i, stop_j)] + np.random.uniform(-300, 300)
        time = time/3600
    else:
        distance = base_travel_distance[(stop_j, stop_i)] + np.random.uniform(-5, 5)
        time = base_travel_time[(stop_j, stop_i)] + np.random.uniform(-300, 300)
        time = time/3600

    # Delay with small variation
    if np.random.rand() < 0.5:  # Add random variation in 10% of cases
        delay = base_delay_time + np.random.uniform(300, 600)
        delay = delay/3600
        event = np.random.choice(delay_reasons[:-1])  # Select a reason excluding "No Delay"
    else:
        delay = base_delay_time + np.random.uniform(-3, 3)
        delay = delay/3600
        event = "No Delay"  # No significant delay

    # Append to lists
    stops_i.append(stop_i)
    stops_j.append(stop_j)
    travel_distance_pattern.append(distance)
    travel_time_pattern.append(time)
    delay_time_pattern.append(delay)
    delay_events.append(event)

# Create a DataFrame with stop pairs and generated values
stop_pairs_pattern = pd.DataFrame({
    'stop_i': stops_i,
    'stop_j': stops_j,
    'travel_distance_km': travel_distance_pattern,
    'travel_time_seconds': travel_time_pattern,
    'delay_time_seconds': delay_time_pattern,
    'delay_event': delay_events
})
# Modify the stop_pairs_pattern DataFrame to include a large delay for route 5 -> 4
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_time_seconds'] = 10 * 3600  # 10 hours delay
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_event'] = 'Road Block'
stop_pairs_pattern.head(30)

,stop_i,stop_j,travel_distance_km,travel_time_seconds,delay_time_seconds,delay_event
0,6,5,27.329856,0.849740,-0.000642,No Delay
1,3,0,46.581670,1.561235,-0.000240,No Delay
2,0,6,64.841317,2.030744,0.000723,No Delay
3,5,2,54.083265,1.838778,-0.000155,No Delay
4,1,5,60.877296,1.985526,0.147347,Road Closed
5,5,3,33.718658,1.024319,0.158381,Accident
6,0,8,65.841899,2.393522,0.127472,Traffic
7,9,6,32.963009,1.037517,0.100361,Weather
8,2,3,31.544489,1.246656,0.000210,No Delay
9,7,4,42.586315,1.282249,0.149699,Road Closed


In [ ]:
import numpy as np
import pandas as pd

num_entries = 2000  # Number of dataset entries
K = 3  # Number of products

# Base travel distances between specific stop pairs
base_travel_distance = {
    (0, 1): 55, (0, 2): 50, (0, 3): 45, (0, 4): 40, (0, 5): 35,
    (0, 6): 60, (0, 7): 65, (0, 8): 70, (0, 9): 75, (0, 10): 80,
    (1, 0): 55, (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60, (1, 6): 65,
    (1, 7): 70, (1, 8): 75, (1, 9): 80, (1, 10): 85,
    (2, 0): 50, (2, 1): 30, (2, 3): 35, (2, 4): 45, (2, 5): 55, (2, 6): 50,
    (2, 7): 55, (2, 8): 60, (2, 9): 65, (2, 10): 70,
    (3, 0): 45, (3, 1): 40, (3, 2): 35, (3, 4): 25, (3, 5): 30, (3, 6): 40,
    (3, 7): 45, (3, 8): 50, (3, 9): 55, (3, 10): 60,
    (4, 0): 40, (4, 1): 50, (4, 2): 45, (4, 3): 25, (4, 5): 20, (4, 6): 35,
    (4, 7): 40, (4, 8): 45, (4, 9): 50, (4, 10): 55,
    (5, 0): 35, (5, 1): 60, (5, 2): 55, (5, 3): 30, (5, 4): 20, (5, 6): 25,
    (5, 7): 30, (5, 8): 35, (5, 9): 40, (5, 10): 45,
    (6, 0): 60, (6, 1): 65, (6, 2): 50, (6, 3): 40, (6, 4): 35, (6, 5): 25,
    (6, 7): 20, (6, 8): 25, (6, 9): 30, (6, 10): 35,
    (7, 0): 65, (7, 1): 70, (7, 2): 55, (7, 3): 45, (7, 4): 40, (7, 5): 30,
    (7, 6): 20, (7, 8): 15, (7, 9): 20, (7, 10): 25,
    (8, 0): 70, (8, 1): 75, (8, 2): 60, (8, 3): 50, (8, 4): 45, (8, 5): 35,
    (8, 6): 25, (8, 7): 15, (8, 9): 10, (8, 10): 15,
    (9, 0): 75, (9, 1): 80, (9, 2): 65, (9, 3): 55, (9, 4): 50, (9, 5): 40,
    (9, 6): 30, (9, 7): 20, (9, 8): 10, (9, 10): 10,
    (10, 0): 80, (10, 1): 85, (10, 2): 70, (10, 3): 60, (10, 4): 55, (10, 5): 45,
    (10, 6): 35, (10, 7): 25, (10, 8): 15, (10, 9): 10,
}

# Add symmetric distances
symmetric_travel_distance = {}
for (i, j), dist in base_travel_distance.items():
    symmetric_travel_distance[(i, j)] = dist
    symmetric_travel_distance[(j, i)] = dist  # Make symmetric

base_travel_distance = symmetric_travel_distance


# Calculate base travel times in seconds
base_travel_time = {k: v / 30*3600  for k, v in base_travel_distance.items()}  # Base time at 50 km/h
base_delay_time = 0  # Base delay time in seconds

# Delay reasons list
delay_reasons = ["Road Closed", "Traffic", "Accident", "Weather", "No Delay"]

# Lists to store generated data
travel_distance_pattern = []
travel_time_pattern = []
delay_time_pattern = []
stops_i = []
stops_j = []
delay_events = []

# Generate shelf life and quality factors
SL_0 = 700 * np.random.rand(K)  # Initial shelf life for each product
SL_r = 100 * np.random.rand(K)  # Required shelf life at destination for each product
Q = np.random.uniform(0.5, 1, K)  # Quality reduction factor for each product

for _ in range(num_entries):
    stop_i = np.random.randint(0, 11)  # Random stop 1-5
    stop_j = np.random.randint(0, 11)

    # Ensure stop_i != stop_j
    while stop_i == stop_j:
        stop_j = np.random.randint(1, 11)

    # Use base travel distance and add some variation
    if (stop_i, stop_j) in base_travel_distance:
        distance = base_travel_distance[(stop_i, stop_j)] + np.random.uniform(-5, 5)
        time = base_travel_time[(stop_i, stop_j)] + np.random.uniform(-300, 300)
        time = time/3600
    else:
        distance = base_travel_distance[(stop_j, stop_i)] + np.random.uniform(-5, 5)
        time = base_travel_time[(stop_j, stop_i)] + np.random.uniform(-300, 300)
        time = time/3600

    # Delay with small variation
    if np.random.rand() < 0.5:  # Add random variation in 10% of cases
        delay = base_delay_time + np.random.uniform(300, 600)
        delay = delay/3600
        event = np.random.choice(delay_reasons[:-1])  # Select a reason excluding "No Delay"
    else:
        delay = base_delay_time + np.random.uniform(-3, 3)
        delay = delay/3600
        event = "No Delay"  # No significant delay

    # Append to lists
    stops_i.append(stop_i)
    stops_j.append(stop_j)
    travel_distance_pattern.append(distance)
    travel_time_pattern.append(time)
    delay_time_pattern.append(delay)
    delay_events.append(event)

# Create a DataFrame with stop pairs and generated values
stop_pairs_pattern = pd.DataFrame({
    'stop_i': stops_i,
    'stop_j': stops_j,
    'travel_distance_km': travel_distance_pattern,
    'travel_time_seconds': travel_time_pattern,
    'delay_time_seconds': delay_time_pattern,
    'delay_event': delay_events
})
# # Modify the stop_pairs_pattern DataFrame to include a large delay for route 5 -> 4
# stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_time_seconds'] = 3600  # 10 hours delay
# stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_event'] = 'Road Block'
# Modify the stop_pairs_pattern DataFrame to include a large delay for route 5 -> 4
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_time_seconds'] = 10 * 3600  # 10 hours delay
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_event'] = 'Road Block'

# Modify the symmetrical route 4 -> 5 as well
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 4) & (stop_pairs_pattern['stop_j'] == 5), 'delay_time_seconds'] = 10 * 3600  # 10 hours delay
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 4) & (stop_pairs_pattern['stop_j'] == 5), 'delay_event'] = 'Road Block'

import numpy as np
import pandas as pd

num_entries = 500  # Number of dataset entries

# Define the types of produce and their typical characteristics
produce_types = ['Apples', 'Bananas', 'Tomatoes', 'xyz']

# Define base shelf life (in hours) for each produce type
base_shelf_life = {
    'Apples': 720,   # 30 days
    'Bananas': 168,  # 7 days
    'Tomatoes': 336  # 14 days
    ,'xyz': 200
}

temperature_range = {
    'Apples': (0, 4),
    'Bananas': (12, 14),
    'Tomatoes': (7, 10)
    ,'xyz': (0, 10)
}

# Lists to store generated data
produce_type_list = []
weight_list = []
shelf_life_list = []
transport_temperature_list = []

for _ in range(num_entries):
    produce = np.random.choice(produce_types)

    weight = np.random.uniform(0.5, 20)

    # Shelf life with some variation (up to ±10%)
    shelf_life = base_shelf_life[produce] + np.random.uniform(-0.1, 0.1) * base_shelf_life[produce]

    # Transportation temperature within defined range for the selected produce
    transport_temperature = np.random.uniform(*temperature_range[produce])

    # Append to lists
    produce_type_list.append(produce)
    weight_list.append(weight)
    shelf_life_list.append(shelf_life)
    transport_temperature_list.append(transport_temperature)

# Create a DataFrame with the generated data
produce_data = pd.DataFrame({
    'produce_type': produce_type_list,
    'weight_kg': weight_list,
    'shelf_life_hours': shelf_life_list,
    'transport_temperature_C': transport_temperature_list
})


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

# =========================== ENVIRONMENT CLASS =========================== #


class DeliveryEnv:
    def __init__(self, stops_data, produce_data, max_steps=50):
        self.stops_data = stops_data
        self.produce_data = produce_data
        self.max_steps = max_steps
        self.all_stops = sorted(set(stops_data['stop_i'].unique()) | set(stops_data['stop_j'].unique()))
        self.n_stops = len(self.all_stops)
        self.action_space_size = self.n_stops
        self.delivery_plan = self.create_delivery_plan()  # New delivery plan
        self.reset()

    def reset(self):
        self.warehouse_location = 0
        self.current_location = self.warehouse_location
        self.current_inventory = {'Apples': 76, 'Bananas': 61, 'Tomatoes': 58, 'xyz': 53}
        self.elapsed_time = 0
        self.shelf_life_remaining = self.initialize_shelf_life()
        self.steps = 0
        self.visited_stops = {self.warehouse_location}
        self.route_log = []
        self.log_initial_stop()
        return self.get_state()

    def initialize_shelf_life(self):
        base_shelf_life = {'Apples': 720, 'Bananas': 168, 'Tomatoes': 336, 'xyz': 200}
        return base_shelf_life.copy()

    def create_delivery_plan(self):
        plan = {
            1: {'Apples': 10, 'Bananas': 5, 'Tomatoes': 5, 'xyz': 0},
            2: {'Apples': 5, 'Bananas': 10, 'Tomatoes': 5, 'xyz': 5},
            3: {'Apples': 5, 'Bananas': 5, 'Tomatoes': 10, 'xyz': 5},
            4: {'Apples': 10, 'Bananas': 5, 'Tomatoes': 0, 'xyz': 10},
            5: {'Apples': 7, 'Bananas': 7, 'Tomatoes': 6, 'xyz': 5},
            6: {'Apples': 8, 'Bananas': 5, 'Tomatoes': 7, 'xyz': 5},
            7: {'Apples': 6, 'Bananas': 8, 'Tomatoes': 6, 'xyz': 5},
            8: {'Apples': 5, 'Bananas': 5, 'Tomatoes': 8, 'xyz': 7},
            9: {'Apples': 10, 'Bananas': 5, 'Tomatoes': 5, 'xyz': 5},
            10: {'Apples': 10, 'Bananas': 6, 'Tomatoes': 6, 'xyz': 6},
        }
        return plan

    def get_state(self):
        return [
            self.current_location,
            self.current_inventory['Apples'],
            self.current_inventory['Bananas'],
            self.current_inventory['Tomatoes'],
            self.current_inventory['xyz'],
            self.shelf_life_remaining['Apples'],
            self.shelf_life_remaining['Bananas'],
            self.shelf_life_remaining['Tomatoes'],
            self.shelf_life_remaining['xyz'],
            self.elapsed_time,
        ]

    def get_nearest_stops(self):
        """Fetch all unvisited stops sorted by distance."""
        unvisited_routes = self.stops_data[
            ((self.stops_data['stop_i'] == self.current_location) & (~self.stops_data['stop_j'].isin(self.visited_stops))) |
            ((self.stops_data['stop_j'] == self.current_location) & (~self.stops_data['stop_i'].isin(self.visited_stops)))
        ]
        if unvisited_routes.empty:
            return []

        unvisited_routes = unvisited_routes.copy()
        unvisited_routes['next_stop'] = np.where(
            unvisited_routes['stop_i'] == self.current_location,
            unvisited_routes['stop_j'],
            unvisited_routes['stop_i']
        )
        return unvisited_routes[['next_stop', 'travel_distance_km', 'travel_time_seconds', 'delay_time_seconds']].values.tolist()
    def get_temperature_adjustment(self):
      """
      Compute the necessary temperature adjustment to maintain ideal storage conditions.
      """

      # Define ideal temperatures
      ideal_temperatures = {'Apples': 2, 'Bananas': 5, 'Tomatoes': 8, 'xyz': 7}

      # Assume the truck currently has a storage temperature (simulated sensor data)
      current_temperatures = {
          'Apples': np.random.uniform(0, 4),  # Simulated sensor reading
          'Bananas': np.random.uniform(12, 14),
          'Tomatoes': np.random.uniform(7, 10),
          'xyz': np.random.uniform(0, 10)
      }

      # Compute temperature deviation for each produce type
      temp_deviation = {p:ideal_temperatures[p] -current_temperatures[p] for p in ideal_temperatures}

      # Compute the average temperature adjustment needed
      avg_temp_adjustment = sum(temp_deviation.values()) / len(temp_deviation)

      return avg_temp_adjustment  # Negative means decrease temp, positive means increase temp


    def select_stop(self, nearest_stops):
      """Select the next stop based on weighted scoring with priority to distance and delay."""
      best_stop = None
      best_score = float('inf')  # Smaller scores are better for prioritizing distance and delay

      for stop in nearest_stops:
          next_stop, travel_distance_km, travel_time, delay_time = stop
          delivery = self.delivery_plan.get(next_stop, {})
          delivery_score = sum(min(self.current_inventory[p], v) for p, v in delivery.items())
          avg_shelf_life = np.mean([self.shelf_life_remaining[p] for p in delivery if delivery[p] > 0])


          # Weighted scoring: prioritize distance, delay, then delivery quantity, then shelf life
          delay_penalty = 50 * (delay_time > 5) + 15 * delay_time  # Large penalty for delays > 5 hours
          score = (10 * travel_distance_km) + delay_penalty - (2 * delivery_score) - avg_shelf_life

          # print(f"Evaluating stop {next_stop}: Distance = {travel_distance_km}, Delay = {delay_time}, "
          #     f"Delivery Score = {delivery_score}, Shelf Life = {avg_shelf_life}, Score = {score}")


          if score < best_score:  # Lower score is better
              best_score = score
              best_stop = stop

      return best_stop


    def step(self, action):
      nearest_stops = self.get_nearest_stops()
      if not nearest_stops:
          if self.current_location != self.warehouse_location:
              self.return_to_warehouse()
          return self.get_state(), 0, True

      # Select best stop based only on shelf life (unchanged)
      best_stop = self.select_stop(nearest_stops)
      if not best_stop:
          if self.current_location != self.warehouse_location:
              self.return_to_warehouse()
          return self.get_state(), 0, True

      next_stop, _, travel_time, delay_time = best_stop

      # Update environment state
      total_time = travel_time + delay_time
      self.elapsed_time += total_time
      self.steps += 1
      self.update_shelf_life()

      # Perform deliveries
      delivery = self.delivery_plan.get(next_stop, {})
      delivered_count = sum(self.update_inventory(p, amt) for p, amt in delivery.items())

      # Compute the temperature adjustment required
      temp_adjustment = self.get_temperature_adjustment()

      # Print temperature adjustment information
      if temp_adjustment < 0:
          print(f"At Stop {next_stop}: Truck should DECREASE temperature by {abs(temp_adjustment):.2f}°C")
      elif temp_adjustment > 0:
          print(f"At Stop {next_stop}: Truck should INCREASE temperature by {temp_adjustment:.2f}°C")
      else:
          print(f"At Stop {next_stop}: No temperature adjustment needed.")

      # Reward calculation (unchanged)
      reward = self.calculate_reward(total_time, delivered_count)

      # Move to next stop
      self.current_location = next_stop
      self.visited_stops.add(next_stop)
      self.log_route(next_stop, delivery, reward)

      # Check if all stops are visited
      if len(self.visited_stops) == self.n_stops:
          self.return_to_warehouse()

      done = self.check_done()
      return self.get_state(), reward, done


    def return_to_warehouse(self):
        """Move back to the warehouse and add travel time."""
        if self.current_location != self.warehouse_location:
            travel_info = self.stops_data[
                (self.stops_data['stop_i'] == self.current_location) & (self.stops_data['stop_j'] == self.warehouse_location)
                | (self.stops_data['stop_j'] == self.current_location) & (self.stops_data['stop_i'] == self.warehouse_location)
            ]
            if not travel_info.empty:
                travel_time = travel_info.iloc[0]['travel_time_seconds']
                self.elapsed_time += travel_time
                self.log_route(self.warehouse_location, {}, 0)  # No penalty for returning to warehouse
            self.current_location = self.warehouse_location

    def update_inventory(self, produce, amount):
        """Update inventory after delivery."""
        delivered = min(self.current_inventory.get(produce, 0), amount)
        self.current_inventory[produce] -= delivered
        return delivered

    def update_shelf_life(self):
        """Update shelf life based on time elapsed and decay rate."""
        decay_rate = 0.05  # Base decay factor
        for produce in self.shelf_life_remaining:
            self.shelf_life_remaining[produce] -= self.elapsed_time * decay_rate
            self.shelf_life_remaining[produce] = max(0, self.shelf_life_remaining[produce])

    def calculate_reward(self, total_time, delivered_produce_count):
        """Calculate reward with separate penalties for travel and delivery rewards."""
        delivery_reward = delivered_produce_count * 10  # Reward for each item delivered
        travel_penalty = total_time * 0.1  # Penalty proportional to travel and delay time
        return delivery_reward - travel_penalty

    def check_done(self):
        """Determine if the episode should terminate."""
        return (
            self.steps >= self.max_steps
            or self.current_location == self.warehouse_location
            and len(self.visited_stops) == self.n_stops
            or all(v == 0 for v in self.current_inventory.values())
            or all(v <= 0 for v in self.shelf_life_remaining.values())
        )

    def handle_no_stops(self):
        if self.current_location != self.warehouse_location:
            self.current_location = self.warehouse_location
        return self.get_state(), -10, True  # Penalty for no valid stops

    def log_route(self, stop, delivery, reward):
        self.route_log.append({'Stop': stop, 'Delivered': delivery, 'Reward': reward})

    def log_initial_stop(self):
        self.route_log.append({'Stop': self.warehouse_location, 'Reward': 0})


# =========================== DQN AGENT =========================== #
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.learning_rate = 0.0005
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = self.build_model().to(self.device)
        self.target_model = self.build_model().to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.update_target_model()

    def build_model(self):
        return nn.Sequential(
            nn.Linear(self.state_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, self.action_size)
        )

    def act(self, state):
        if random.random() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        return torch.argmax(self.model(state)).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*minibatch)

        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        q_values = self.model(states).gather(1, actions.unsqueeze(1)).squeeze()
        max_next_q_values = self.target_model(next_states).max(1)[0].detach()
        target_q_values = rewards + (1 - dones) * self.gamma * max_next_q_values

        loss = nn.MSELoss()(q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())


# =========================== TRAINING LOOP =========================== #
def train_dqn(env, episodes=500, batch_size=64):
    agent = DQNAgent(len(env.get_state()), env.action_space_size)
    for episode in range(episodes):
        state = env.reset()
        total_reward, done = 0, False

        while not done:
            action = agent.act(state)
            next_state, reward, done = env.step(action)
            agent.memory.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward

            if len(agent.memory) > batch_size:
                agent.replay(batch_size)

       #print(f"Episode {episode}, Total Reward: {total_reward:.2f}, Epsilon: {agent.epsilon:.4f}")

    return agent

import matplotlib.pyplot as plt
import torch
import os

def save_model(agent, file_path="dqn_agent.pth"):
    """Save the trained model to a file."""
    torch.save(agent.model.state_dict(), file_path)
    print(f"Model saved to {file_path}")

def load_model(agent, file_path="dqn_agent.pth"):
    """Load a trained model from a file."""
    if os.path.exists(file_path):
        agent.model.load_state_dict(torch.load(file_path))
        agent.update_target_model()
        print(f"Model loaded from {file_path}")
    else:
        print(f"Model file {file_path} does not exist.")

def test_dqn_training(stop_data, produce_data, episodes=300, save_path="dqn_agent.pth"):
    # Initialize environment

    env = DeliveryEnv(stops_data=stop_data, produce_data=produce_data, max_steps=50)

    # Train DQN agent
    rewards = []  # Track rewards for visualization
    elapsed_times = []  # Track elapsed timesa
    agent = DQNAgent(len(env.get_state()), env.action_space_size)

    print("Starting DQN training...\n")
    for episode in range(episodes):
        state = env.reset()
        total_reward, done = 0, False
        elapsed_time = 0

        while not done:
            action = agent.act(state)
            next_state, reward, done = env.step(action)
            agent.memory.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward
            elapsed_time = env.elapsed_time  # Track final elapsed time

            if len(agent.memory) > 64:  # Start replay when enough experience
                agent.replay(64)

        rewards.append(total_reward)
        elapsed_times.append(elapsed_time)

        # if (episode + 1) % 10 == 0:
        #     print(f"Episode {episode + 1}: Total Reward = {total_reward:.2f}, Elapsed Time = {elapsed_time:.2f} hours, Epsilon = {agent.epsilon:.4f}")

    # Update target model after training
    agent.update_target_model()

    # Save the trained model
    save_model(agent, save_path)


    # ================= PRINT BEST ROUTE ================= #
    print("\n--- Best Route Found ---")
    env.reset()
    done = False
    while not done:
        action = agent.act(env.get_state())
        _, _, done = env.step(action)

    # Display route log
    for log in env.route_log:
        print(log)

# ================= RUN THE TEST ================= #
test_dqn_training(stop_pairs_pattern, produce_data, episodes=100)


Starting DQN training...

At Stop 5.0: Truck should DECREASE temperature by 3.00°C
At Stop 6.0: Truck should DECREASE temperature by 2.28°C
At Stop 7.0: Truck should DECREASE temperature by 2.21°C
At Stop 8.0: Truck should DECREASE temperature by 0.17°C
At Stop 9.0: Truck should DECREASE temperature by 3.00°C
At Stop 10.0: Truck should DECREASE temperature by 1.34°C
At Stop 4.0: Truck should DECREASE temperature by 1.13°C
At Stop 3.0: Truck should DECREASE temperature by 1.47°C
At Stop 2.0: Truck should DECREASE temperature by 2.91°C
At Stop 1.0: Truck should DECREASE temperature by 1.60°C
At Stop 5.0: Truck should DECREASE temperature by 2.60°C
At Stop 6.0: Truck should DECREASE temperature by 0.92°C
At Stop 7.0: Truck should DECREASE temperature by 2.04°C
At Stop 8.0: Truck should DECREASE temperature by 0.62°C
At Stop 9.0: Truck should DECREASE temperature by 0.63°C
At Stop 10.0: Truck should DECREASE temperature by 1.04°C
At Stop 4.0: Truck should DECREASE temperature by 1.85°C
At 

# Hybrid Code

In [ ]:
import numpy as np
import pandas as pd

num_entries = 2000  # Number of dataset entries
K = 3  # Number of products

# Base travel distances between specific stop pairs
base_travel_distance = {
    (0, 1): 55, (0, 2): 50, (0, 3): 45, (0, 4): 40, (0, 5): 35,
    (0, 6): 60, (0, 7): 65, (0, 8): 70, (0, 9): 75, (0, 10): 80,
    (1, 0): 55, (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60, (1, 6): 65,
    (1, 7): 70, (1, 8): 75, (1, 9): 80, (1, 10): 85,
    (2, 0): 50, (2, 1): 30, (2, 3): 35, (2, 4): 45, (2, 5): 55, (2, 6): 50,
    (2, 7): 55, (2, 8): 60, (2, 9): 65, (2, 10): 70,
    (3, 0): 45, (3, 1): 40, (3, 2): 35, (3, 4): 25, (3, 5): 30, (3, 6): 40,
    (3, 7): 45, (3, 8): 50, (3, 9): 55, (3, 10): 60,
    (4, 0): 40, (4, 1): 50, (4, 2): 45, (4, 3): 25, (4, 5): 20, (4, 6): 35,
    (4, 7): 40, (4, 8): 45, (4, 9): 50, (4, 10): 55,
    (5, 0): 35, (5, 1): 60, (5, 2): 55, (5, 3): 30, (5, 4): 20, (5, 6): 25,
    (5, 7): 30, (5, 8): 35, (5, 9): 40, (5, 10): 45,
    (6, 0): 60, (6, 1): 65, (6, 2): 50, (6, 3): 40, (6, 4): 35, (6, 5): 25,
    (6, 7): 20, (6, 8): 25, (6, 9): 30, (6, 10): 35,
    (7, 0): 65, (7, 1): 70, (7, 2): 55, (7, 3): 45, (7, 4): 40, (7, 5): 30,
    (7, 6): 20, (7, 8): 15, (7, 9): 20, (7, 10): 25,
    (8, 0): 70, (8, 1): 75, (8, 2): 60, (8, 3): 50, (8, 4): 45, (8, 5): 35,
    (8, 6): 25, (8, 7): 15, (8, 9): 10, (8, 10): 15,
    (9, 0): 75, (9, 1): 80, (9, 2): 65, (9, 3): 55, (9, 4): 50, (9, 5): 40,
    (9, 6): 30, (9, 7): 20, (9, 8): 10, (9, 10): 10,
    (10, 0): 80, (10, 1): 85, (10, 2): 70, (10, 3): 60, (10, 4): 55, (10, 5): 45,
    (10, 6): 35, (10, 7): 25, (10, 8): 15, (10, 9): 10,
}

# Add symmetric distances
symmetric_travel_distance = {}
for (i, j), dist in base_travel_distance.items():
    symmetric_travel_distance[(i, j)] = dist
    symmetric_travel_distance[(j, i)] = dist  # Make symmetric

base_travel_distance = symmetric_travel_distance


# Calculate base travel times in seconds
base_travel_time = {k: v / 30*3600  for k, v in base_travel_distance.items()}  # Base time at 50 km/h
base_delay_time = 0  # Base delay time in seconds

# Delay reasons list
delay_reasons = ["Road Closed", "Traffic", "Accident", "Weather", "No Delay"]

# Lists to store generated data
travel_distance_pattern = []
travel_time_pattern = []
delay_time_pattern = []
stops_i = []
stops_j = []
delay_events = []

# Generate shelf life and quality factors
SL_0 = 700 * np.random.rand(K)  # Initial shelf life for each product
SL_r = 100 * np.random.rand(K)  # Required shelf life at destination for each product
Q = np.random.uniform(0.5, 1, K)  # Quality reduction factor for each product

for _ in range(num_entries):
    stop_i = np.random.randint(0, 11)  # Random stop 1-5
    stop_j = np.random.randint(0, 11)

    # Ensure stop_i != stop_j
    while stop_i == stop_j:
        stop_j = np.random.randint(1, 11)

    # Use base travel distance and add some variation
    if (stop_i, stop_j) in base_travel_distance:
        distance = base_travel_distance[(stop_i, stop_j)] + np.random.uniform(-5, 5)
        time = base_travel_time[(stop_i, stop_j)] + np.random.uniform(-300, 300)
        time = time/3600
    else:
        distance = base_travel_distance[(stop_j, stop_i)] + np.random.uniform(-5, 5)
        time = base_travel_time[(stop_j, stop_i)] + np.random.uniform(-300, 300)
        time = time/3600

    # Delay with small variation
    if np.random.rand() < 0.1:  # Add random variation in 10% of cases
        delay = base_delay_time + np.random.uniform(1, 2)
        delay = delay/3600
        event = np.random.choice(delay_reasons[:-1])  # Select a reason excluding "No Delay"
    else:
        delay = base_delay_time + np.random.uniform(-1, 1)
        delay = delay/3600
        event = "No Delay"  # No significant delay

    # Append to lists
    stops_i.append(stop_i)
    stops_j.append(stop_j)
    travel_distance_pattern.append(distance)
    travel_time_pattern.append(time)
    delay_time_pattern.append(delay)
    delay_events.append(event)

# Create a DataFrame with stop pairs and generated values
stop_pairs_pattern = pd.DataFrame({
    'stop_i': stops_i,
    'stop_j': stops_j,
    'travel_distance_km': travel_distance_pattern,
    'travel_time_seconds': travel_time_pattern,
    'delay_time_seconds': delay_time_pattern,
    'delay_event': delay_events
})
# # # Modify the stop_pairs_pattern DataFrame to include a large delay for route 5 -> 4
# # stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_time_seconds'] = 3600  # 10 hours delay
# # stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 5) & (stop_pairs_pattern['stop_j'] == 4), 'delay_event'] = 'Road Block'
# # # Modify the stop_pairs_pattern DataFrame to include a large delay for route 5 -> 4
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 3) & (stop_pairs_pattern['stop_j'] == 5), 'delay_time_seconds'] = 10 * 3600  # 10 hours delay
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 3) & (stop_pairs_pattern['stop_j'] == 5), 'delay_event'] = 'Road Block'

# Modify the symmetrical route 4 -> 5 as well
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 3) & (stop_pairs_pattern['stop_j'] == 5), 'delay_time_seconds'] = 10 * 3600  # 10 hours delay
stop_pairs_pattern.loc[(stop_pairs_pattern['stop_i'] == 3) & (stop_pairs_pattern['stop_j'] == 5), 'delay_event'] = 'Road Block'

# import numpy as np
import pandas as pd

num_entries = 500  # Number of dataset entries

# Define the types of produce and their typical characteristics
produce_types = ['Apples', 'Bananas', 'Tomatoes', 'xyz']

# Define base shelf life (in hours) for each produce type
base_shelf_life = {
    'Apples': 720,   # 30 days
    'Bananas': 168,  # 7 days
    'Tomatoes': 336  # 14 days
    ,'xyz': 200
}

temperature_range = {
    'Apples': (0, 4),
    'Bananas': (12, 14),
    'Tomatoes': (7, 10)
    ,'xyz': (0, 10)
}

# Lists to store generated data
produce_type_list = []
weight_list = []
shelf_life_list = []
transport_temperature_list = []

for _ in range(num_entries):
    produce = np.random.choice(produce_types)

    weight = np.random.uniform(0.5, 20)

    # Shelf life with some variation (up to ±10%)
    shelf_life = base_shelf_life[produce] + np.random.uniform(-0.1, 0.1) * base_shelf_life[produce]

    # Transportation temperature within defined range for the selected produce
    transport_temperature = np.random.uniform(*temperature_range[produce])

    # Append to lists
    produce_type_list.append(produce)
    weight_list.append(weight)
    shelf_life_list.append(shelf_life)
    transport_temperature_list.append(transport_temperature)

# Create a DataFrame with the generated data
produce_data = pd.DataFrame({
    'produce_type': produce_type_list,
    'weight_kg': weight_list,
    'shelf_life_hours': shelf_life_list,
    'transport_temperature_C': transport_temperature_list
})


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

# =========================== ENVIRONMENT CLASS =========================== #


class DeliveryEnv:
    def __init__(self, stops_data, produce_data, max_steps=50):
        self.stops_data = stops_data
        self.produce_data = produce_data
        self.max_steps = max_steps
        self.all_stops = sorted(set(stops_data['stop_i'].unique()) | set(stops_data['stop_j'].unique()))
        self.n_stops = len(self.all_stops)
        self.action_space_size = self.n_stops
        self.delivery_plan = self.create_delivery_plan()
        self.predefined_path = [0, 1, 2, 3, 5, 9, 10, 8, 7, 6, 4, 0]  # Predefined path
        self.reset()

    def reset(self):
        self.warehouse_location = 0
        self.current_location = self.warehouse_location
        self.current_inventory = {'Apples': 76, 'Bananas': 61, 'Tomatoes': 58, 'xyz': 53}
        self.elapsed_time = 0
        self.shelf_life_remaining = self.initialize_shelf_life()
        self.steps = 0
        self.visited_stops = {self.warehouse_location}
        self.route_log = []
        self.path_index = 0  # Track current index in the predefined path
        self.log_initial_stop()
        return self.get_state()
    def create_delivery_plan(self):
        plan = {
            1: {'Apples': 10, 'Bananas': 5, 'Tomatoes': 5, 'xyz': 0},
            2: {'Apples': 5, 'Bananas': 10, 'Tomatoes': 5, 'xyz': 5},
            3: {'Apples': 5, 'Bananas': 5, 'Tomatoes': 10, 'xyz': 5},
            4: {'Apples': 10, 'Bananas': 5, 'Tomatoes': 0, 'xyz': 10},
            5: {'Apples': 7, 'Bananas': 7, 'Tomatoes': 6, 'xyz': 5},
            6: {'Apples': 8, 'Bananas': 5, 'Tomatoes': 7, 'xyz': 5},
            7: {'Apples': 6, 'Bananas': 8, 'Tomatoes': 6, 'xyz': 5},
            8: {'Apples': 5, 'Bananas': 5, 'Tomatoes': 8, 'xyz': 7},
            9: {'Apples': 10, 'Bananas': 5, 'Tomatoes': 5, 'xyz': 5},
            10: {'Apples': 10, 'Bananas': 6, 'Tomatoes': 6, 'xyz': 6},
        }
        return plan

    def get_state(self):
        return [
            self.current_location,
            self.current_inventory['Apples'],
            self.current_inventory['Bananas'],
            self.current_inventory['Tomatoes'],
            self.current_inventory['xyz'],
            self.shelf_life_remaining['Apples'],
            self.shelf_life_remaining['Bananas'],
            self.shelf_life_remaining['Tomatoes'],
            self.shelf_life_remaining['xyz'],
            self.elapsed_time,
        ]
    def select_stop(self, nearest_stops):
      """Select the next stop based on weighted scoring with priority to distance and delay."""
      best_stop = None
      best_score = float('inf')  # Smaller scores are better for prioritizing distance and delay

      for stop in nearest_stops:
          next_stop, travel_distance_km, travel_time, delay_time = stop
          delivery = self.delivery_plan.get(next_stop, {})
          delivery_score = sum(min(self.current_inventory[p], v) for p, v in delivery.items())
          avg_shelf_life = np.mean([self.shelf_life_remaining[p] for p in delivery if delivery[p] > 0])


          # Weighted scoring: prioritize distance, delay, then delivery quantity, then shelf life
          delay_penalty = 50 * (delay_time > 5) + 15 * delay_time  # Large penalty for delays > 5 hours
          score = (10 * travel_distance_km) + delay_penalty - (2 * delivery_score) - avg_shelf_life

          # print(f"Evaluating stop {next_stop}: Distance = {travel_distance_km}, Delay = {delay_time}, "
          #     f"Delivery Score = {delivery_score}, Shelf Life = {avg_shelf_life}, Score = {score}")


          if score < best_score:  # Lower score is better
              best_score = score
              best_stop = stop

      return best_stop

    def initialize_shelf_life(self):
        base_shelf_life = {'Apples': 720, 'Bananas': 168, 'Tomatoes': 336, 'xyz': 200}
        return base_shelf_life.copy()

    def step(self, action):
      if self.path_index < len(self.predefined_path) - 1:
          # Follow the predefined path
          next_stop = self.predefined_path[self.path_index + 1]
          route_info = self.stops_data[
              ((self.stops_data['stop_i'] == self.current_location) & (self.stops_data['stop_j'] == next_stop)) |
              ((self.stops_data['stop_j'] == self.current_location) & (self.stops_data['stop_i'] == next_stop))
          ]

          if not route_info.empty:
              travel_time = route_info.iloc[0]['travel_time_seconds']
              delay_time = route_info.iloc[0]['delay_time_seconds']

              # Check if delay is significant (>1 hour)
              if delay_time < 3600:  # 1 hour in seconds
                  self.path_index += 1
                  return self.execute_stop(next_stop, travel_time + delay_time)

      # If there's a significant delay or at the end of the path, revert to RL logic
      nearest_stops = self.get_nearest_stops()
      if not nearest_stops:
          self.return_to_warehouse()
          return self.get_state(), 0, True

      best_stop = self.select_stop(nearest_stops)
      if not best_stop:
          self.return_to_warehouse()
          return self.get_state(), 0, True

      next_stop, _, travel_time, delay_time = best_stop
      return self.execute_stop(next_stop, travel_time + delay_time)

    def execute_stop(self, next_stop, total_time):
        self.elapsed_time += total_time
        self.steps += 1
        self.update_shelf_life()

        delivery = self.delivery_plan.get(next_stop, {})
        delivered_count = sum(self.update_inventory(p, amt) for p, amt in delivery.items())
        reward = self.calculate_reward(total_time, delivered_count)

        self.current_location = next_stop
        self.visited_stops.add(next_stop)
        self.log_route(next_stop, delivery, reward)

        # Check if all stops are visited and return to warehouse
        if len(self.visited_stops) == self.n_stops:
            self.return_to_warehouse()

        done = self.check_done()
        return self.get_state(), reward, done
    def get_nearest_stops(self):
        """Fetch all unvisited stops sorted by distance."""
        unvisited_routes = self.stops_data[
            ((self.stops_data['stop_i'] == self.current_location) & (~self.stops_data['stop_j'].isin(self.visited_stops))) |
            ((self.stops_data['stop_j'] == self.current_location) & (~self.stops_data['stop_i'].isin(self.visited_stops)))
        ]
        if unvisited_routes.empty:
            return []

        unvisited_routes = unvisited_routes.copy()
        unvisited_routes['next_stop'] = np.where(
            unvisited_routes['stop_i'] == self.current_location,
            unvisited_routes['stop_j'],
            unvisited_routes['stop_i']
        )
        return unvisited_routes[['next_stop', 'travel_distance_km', 'travel_time_seconds', 'delay_time_seconds']].values.tolist()

    def return_to_warehouse(self):
        if self.current_location != self.warehouse_location:
            travel_info = self.stops_data[
                ((self.stops_data['stop_i'] == self.current_location) & (self.stops_data['stop_j'] == self.warehouse_location)) |
                ((self.stops_data['stop_j'] == self.current_location) & (self.stops_data['stop_i'] == self.warehouse_location))
            ]
            if not travel_info.empty:
                travel_time = travel_info.iloc[0]['travel_time_seconds']
                self.elapsed_time += travel_time
                self.log_route(self.warehouse_location, {}, 0)
            self.current_location = self.warehouse_location

    def update_inventory(self, produce, amount):
        """Update inventory after delivery."""
        delivered = min(self.current_inventory.get(produce, 0), amount)
        self.current_inventory[produce] -= delivered
        return delivered

    def update_shelf_life(self):
        """Update shelf life based on time elapsed and decay rate."""
        decay_rate = 0.05  # Base decay factor
        for produce in self.shelf_life_remaining:
            self.shelf_life_remaining[produce] -= self.elapsed_time * decay_rate
            self.shelf_life_remaining[produce] = max(0, self.shelf_life_remaining[produce])

    def calculate_reward(self, total_time, delivered_produce_count):
        """Calculate reward with separate penalties for travel and delivery rewards."""
        delivery_reward = delivered_produce_count * 10  # Reward for each item delivered
        travel_penalty = total_time * 0.1  # Penalty proportional to travel and delay time
        return delivery_reward - travel_penalty

    def check_done(self):
        """Determine if the episode should terminate."""
        return (
            self.steps >= self.max_steps
            or self.current_location == self.warehouse_location
            and len(self.visited_stops) == self.n_stops
            or all(v == 0 for v in self.current_inventory.values())
            or all(v <= 0 for v in self.shelf_life_remaining.values())
        )

    def handle_no_stops(self):
        if self.current_location != self.warehouse_location:
            self.current_location = self.warehouse_location
        return self.get_state(), -10, True  # Penalty for no valid stops

    def log_route(self, stop, delivery, reward):
        self.route_log.append({'Stop': stop, 'Delivered': delivery, 'Reward': reward})

    def log_initial_stop(self):
        self.route_log.append({'Stop': self.warehouse_location, 'Reward': 0})


# =========================== DQN AGENT =========================== #
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.learning_rate = 0.0005
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = self.build_model().to(self.device)
        self.target_model = self.build_model().to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.update_target_model()

    def build_model(self):
        return nn.Sequential(
            nn.Linear(self.state_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, self.action_size)
        )

    def act(self, state):
        if random.random() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        return torch.argmax(self.model(state)).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*minibatch)

        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        q_values = self.model(states).gather(1, actions.unsqueeze(1)).squeeze()
        max_next_q_values = self.target_model(next_states).max(1)[0].detach()
        target_q_values = rewards + (1 - dones) * self.gamma * max_next_q_values

        loss = nn.MSELoss()(q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())


# =========================== TRAINING LOOP =========================== #
def train_dqn(env, episodes=500, batch_size=64):
    agent = DQNAgent(len(env.get_state()), env.action_space_size)
    for episode in range(episodes):
        state = env.reset()
        total_reward, done = 0, False

        while not done:
            action = agent.act(state)
            next_state, reward, done = env.step(action)
            agent.memory.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward

            if len(agent.memory) > batch_size:
                agent.replay(batch_size)

        print(f"Episode {episode}, Total Reward: {total_reward:.2f}, Epsilon: {agent.epsilon:.4f}")

    return agent

import matplotlib.pyplot as plt
import torch
import os

def save_model(agent, file_path="dqn_agent.pth"):
    """Save the trained model to a file."""
    torch.save(agent.model.state_dict(), file_path)
    print(f"Model saved to {file_path}")

def load_model(agent, file_path="dqn_agent.pth"):
    """Load a trained model from a file."""
    if os.path.exists(file_path):
        agent.model.load_state_dict(torch.load(file_path))
        agent.update_target_model()
        print(f"Model loaded from {file_path}")
    else:
        print(f"Model file {file_path} does not exist.")

def test_dqn_training(stop_data, produce_data, episodes=300, save_path="dqn_agent.pth"):
    # Initialize environment

    env = DeliveryEnv(stops_data=stop_data, produce_data=produce_data, max_steps=50)

    # Train DQN agent
    rewards = []  # Track rewards for visualization
    elapsed_times = []  # Track elapsed times
    agent = DQNAgent(len(env.get_state()), env.action_space_size)

    print("Starting DQN training...\n")
    for episode in range(episodes):
        state = env.reset()
        total_reward, done = 0, False
        elapsed_time = 0

        while not done:
            action = agent.act(state)
            next_state, reward, done = env.step(action)
            agent.memory.append((state, action, reward, next_state, done))
            state = next_state
            total_reward += reward
            elapsed_time = env.elapsed_time  # Track final elapsed time

            if len(agent.memory) > 64:  # Start replay when enough experience
                agent.replay(64)

        rewards.append(total_reward)
        elapsed_times.append(elapsed_time)

        if (episode + 1) % 10 == 0:
            print(f"Episode {episode + 1}: Total Reward = {total_reward:.2f}, Elapsed Time = {elapsed_time:.2f} hours, Epsilon = {agent.epsilon:.4f}")

    # Update target model after training
    agent.update_target_model()

    # Save the trained model
    save_model(agent, save_path)


    # ================= PRINT BEST ROUTE ================= #
    print("\n--- Best Route Found ---")
    env.reset()
    done = False
    while not done:
        action = agent.act(env.get_state())
        _, _, done = env.step(action)

    # Display route log
    for log in env.route_log:
        print(log)

# ================= RUN THE TEST ================= #
test_dqn_training(stop_pairs_pattern, produce_data, episodes=100)


Starting DQN training...

Episode 10: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.8349
Episode 20: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.5058
Episode 30: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.3064
Episode 40: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.1856
Episode 50: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.1124
Episode 60: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.0681
Episode 70: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.0413
Episode 80: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.0250
Episode 90: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.0151
Episode 100: Total Reward = 2479.09, Elapsed Time = 11.06 hours, Epsilon = 0.0100
Model saved to dqn_agent.pth

--- Best Route Found ---
{'Stop': 0, 'Reward': 0}
{'Stop': 1, 'Delivered': {'Apples': 10, 'Bananas': 5, 'Tomatoes': 5, 'xyz': 0}, 'Re